In [1]:
pip install happybase

   ---------------------------------------- 0.0/846.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/846.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/846.9 kB ? eta -:--:--
   ------------ --------------------------- 262.1/846.9 kB ? eta -:--:--
   ------------------------ --------------- 524.3/846.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 846.9/846.9 kB 1.6 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [thriftpy2]
   -------------------- ------------------- 2/4 [importlib-resources]
   ---------------------------------------- 4/4 [happybase]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import happybase
import json
import time
import os

def load_and_ingest_data(json_file_path):
    # 1. Connect to the Dockerized HBase Thrift server
    try:
        connection = happybase.Connection('localhost', port=9090)
        connection.open()
        print("Connected to HBase successfully.")
    except Exception as e:
        print(f"Error connecting to HBase: {e}")
        print("Please ensure your Docker container is running and port 9090 is exposed.")
        return

    # 2. Bind to your configured tables
    browsing_table = connection.table('user_browsing_data')
    metrics_table = connection.table('product_performance')

    # 3. Check if the target file exists
    if not os.path.exists(json_file_path):
        print(f"Error: The file '{json_file_path}' was not found.")
        print("Please ensure you provide the correct path to your generated dataset.")
        return

    print(f"Reading records from {json_file_path}...")
    
    with open(json_file_path, 'r') as f:
        try:
            records = json.load(f)
        except json.JSONDecodeError:
            # Handle line-by-line JSON format if files are structured that way
            f.seek(0)
            records = [json.loads(line) for line in f if line.strip()]

    print(f"Loaded {len(records)} records. Commencing batch ingestion...")

    # Use batches to optimize network throughput to Docker
    with browsing_table.batch(batch_size=100) as browsing_batch:
        for idx, record in enumerate(records):
            # Extract standard session attributes
            user_id = str(record.get("user_id"))
            session_id = str(record.get("session_id"))
            duration = str(record.get("duration_seconds", record.get("duration", "0")))
            
            # Extract nested device profile
            device_info = record.get("device_profile", {})
            device_type = str(device_info.get("type", record.get("device", "unknown")))
            
            # Handle products list
            viewed_prods = record.get("viewed_products", [])
            products_str = ",".join(viewed_prods)

            # --- RATIONALE: TIME-SERIES ROW KEY GENERATION ---
            # Extract or simulate an ISO timestamp (e.g., '2026-06-26T14:30:00')
            timestamp_str = record.get("start_time", record.get("timestamp", "2026-06-26T12:00:00"))
            
            # Parse time string cleanly to epoch integer
            try:
                clean_ts = timestamp_str.split(".")[0].replace("Z", "")
                epoch_time = int(time.mktime(time.strptime(clean_ts, "%Y-%m-%dT%H:%M:%S")))
            except Exception:
                epoch_time = int(time.time()) # Fallback to current time if format mismatches

            # Compute reverse timestamp: Long.MAX_VALUE - epoch_time
            reverse_ts = str(9223372036854775807 - epoch_time)
            
            # Structured Row Key ensures latest entries surface first during scans
            browsing_row_key = f"{user_id}_{reverse_ts}"

            # 4. Stage the user browsing metrics payload
            browsing_payload = {
                b'cf_session:session_id': session_id.encode(),
                b'cf_session:duration': duration.encode(),
                b'cf_session:device': device_type.encode(),
                b'cf_session:products': products_str.encode()
            }
            browsing_batch.put(browsing_row_key.encode(), browsing_payload)

            # --- RATIONALE: PRODUCT METRICS TRACKING ---
            # Key layout: product_id + date (YYYY-MM-DD)
            date_partition = timestamp_str.split("T")[0]
            for prod_id in viewed_prods:
                metrics_row_key = f"{prod_id}_{date_partition}"
                # Use counter increments to prevent write collisions across parallel session processes
                try:
                    metrics_table.counter_inc(metrics_row_key.encode(), b'cf_metrics:views_count', value=1)
                except Exception:
                    pass # Safely jump record errors

            if (idx + 1) % 500 == 0:
                print(f" -> Processed {idx + 1} session records...")

    print("\nData ingestion into HBase completed successfully!")
    connection.close()

if __name__ == "__main__":
    # Pointing directly to the first representative large subset block
    DATASET_PATH = r"C:\Users\DLT\e-commerce\data\sessions_0.json"
    load_and_ingest_data(DATASET_PATH)


Connected to HBase successfully.
Reading records from C:\Users\DLT\e-commerce\data\sessions_0.json...
Loaded 100000 records. Commencing batch ingestion...
 -> Processed 500 session records...
 -> Processed 1000 session records...
 -> Processed 1500 session records...
 -> Processed 2000 session records...
 -> Processed 2500 session records...
 -> Processed 3000 session records...
 -> Processed 3500 session records...
 -> Processed 4000 session records...
 -> Processed 4500 session records...
 -> Processed 5000 session records...
 -> Processed 5500 session records...
 -> Processed 6000 session records...
 -> Processed 6500 session records...
 -> Processed 7000 session records...
 -> Processed 7500 session records...
 -> Processed 8000 session records...
 -> Processed 8500 session records...
 -> Processed 9000 session records...
 -> Processed 9500 session records...
 -> Processed 10000 session records...
 -> Processed 10500 session records...
 -> Processed 11000 session records...
 -> Pro